# R2lab F1 handover between two OAI DUs with Benetel RUs (O-RAN 7.2)

## **5G Deployment Example With R2lab:**

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UDM, UDR, UPF
- OAI split RAN: CU-CP and CU-UP on the VM `node2`, and 2 DUs on two 100 MHz carriers 200 MHz apart (inter-frequency): `du01` on `sopnode-f3` with the Benetel RAN550 `benetel1`, `du02` on `sopnode-f4` with the Benetel RAN550 `benetel2`, both over the O-RAN 7.2 fronthaul (DPDK on SR-IOV VFs, PTP time synchronization)
- COTS UEs: `qhat01` and `qhat02` in R2Lab, attached to `du01` one after the other, then `qhat01` is handed over to `du02` with the CU-CP telnet command; `qhat03` stays disconnected (it does not attach to `du01` alone, and `benetel2` saturates it)
- Core on the VM `node10`, monitoring on the VM `node11`; CSI: `du01` logs SRS channel estimates per RB, shown live on `node11` and collected per traffic section

The radio and fronthaul parameters are those of the 5g_ansible Benetel setup: https://github.com/sopnode/oai5g-rru/blob/main/rru/du-config-benetel.yaml, `rru/benetel1.env` and `rru/benetel2.env`; `du02` uses the next 100 MHz carrier. No RIC is deployed: the handover is triggered from the CU-CP.

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal servers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
#! slices auth login

Install IFRAN:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../..").resolve()
# This repository (r2lab_experiments: O-RAN 7.2 charts and templates, handover trigger) and NADOR as published
# (nador==1.1.1, from PyPI): install it in editable mode.
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
# IFRAN develop (UE configure, RU SSH access, kubelet parameters, miscellaneous data in the rendered files), not yet in
# an IFRAN release: the O-RAN 7.2 node setup and the Benetel RU configuration are playbooks of this repository.
# Install it after this repository, whose NADOR requirement (ifran==0.0.6) would otherwise replace it.
# To test unpushed IFRAN changes, set IFRAN to the path of a local clone. Do not install it in editable
# mode (-e): the IFRAN Ansible roles are only found in an installed package.
IFRAN = "ifran @ git+https://github.com/ifran-project/ifran.git@develop"
# IFRAN = "/Users/<user>/git/post-5g/ifran"
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "{IFRAN}"
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


Install the Ansible dependencies:

In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r ../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

IFRAN keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

Edit the SLICES project, the VM infrastructure, flavor and lifetime here. Keep the SLICES CLI authenticated in this environment.

In [ ]:
from datetime import UTC, datetime

# VMs of the experiment (BI provisioner); the experiment name includes a timestamp, so that a run never reuses
# (and deletes) the VMs of another one
PROJECT = "post5g-beta"
EXPERIMENT = "oai-benetel-2du-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
INFRA = "fr-sophia2-bi-vm1"
VM_FLAVOR = "large"
VM_DURATION = "8h"  # VM lifetime
VM_NODES = ["node1", "node2", "node10", "node11"]
print(f"Experiment: {EXPERIMENT}")


In [ ]:
from metamodel_3gpp.models.metamodel import MetaModel

# Inter-frequency F1 handover: two 100 MHz carriers (273 PRB). On a single carrier, the two RUs in the
# same room interfere: the UEs lose the link and re-establish on either cell, and a handover towards the weaker
# cell fails (the UE does not decode the target RAR).
# du01 keeps the carrier of the 5g_ansible Benetel setup (rru/du-config-benetel.yaml), du02 is 298.08 MHz above
# (207 SSB raster steps, same SSB and CORESET#0 offsets from point A), at the top of n78 (3699.66-3797.94 MHz):
# ~200 MHz between the carriers. With adjacent carriers (2.5 MHz apart), du02 in a bad state after a crash
# jammed the uplink of du01 (2026-11-19: false PRACH detections, out-of-sync UEs). TDD 2.5 ms: 3 DL slots + 6 DL symbols,
# 1 UL slot + 4 UL symbols.
metamodel = MetaModel.from_yaml(
"""
rus:
  benetel1:
    type: real
    timeSource: external
    clockSource: external
    numberOfTxPorts: 4
    numberOfRxPorts: 4
    txAttenuationDB: 0
    rxAttenuationDB: 0
    rxGainDB: 75
  benetel2:
    type: real
    timeSource: external
    clockSource: external
    numberOfTxPorts: 4
    numberOfRxPorts: 4
    txAttenuationDB: 0
    rxAttenuationDB: 0
    rxGainDB: 75
ran:
  components:
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
      config:
        cu_cp: cucp
    du01:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 100
          pointAFrequencyArfcn: 626772  # 3401.58 MHz, carrier 3401.58-3499.86 MHz
          ssbArfcn: 630048  # 3450.72 MHz
          ssbPbchBlockPowerDbm: -15
          prachConfigurationIndex: 152
          msg1FrequencyStartPrb: 0
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 3
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 1
            nrofUplinkSymbols: 4
        gnbDuId: 1
        nrCellId: 1
        physicalCellId: 0
        ru: benetel1
        cu_cp: cucp
        cu_up: cuup
    du02:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 100
          pointAFrequencyArfcn: 646644  # 3699.66 MHz, carrier 3699.66-3797.94 MHz
          ssbArfcn: 649920  # 3748.80 MHz
          ssbPbchBlockPowerDbm: -15
          prachConfigurationIndex: 152
          msg1FrequencyStartPrb: 0
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 3
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 1
            nrofUplinkSymbols: 4
        gnbDuId: 2
        nrCellId: 2
        physicalCellId: 1
        ru: benetel2
        cu_cp: cucp
        cu_up: cuup
ric:
  components: {}
ues:
  qhat01:
    type: cots
    config:
      supi: '001010000000006'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - du01
  qhat02:
    type: cots
    config:
      supi: '001010000000007'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - du01
  qhat03:
    type: cots
    config:
      supi: '001010000000008'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - du01
"""
)


To see the full content of this Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

Four VMs (BI provisioner, `INFRA` above): `node1` runs the Kubernetes control plane and the iperf server, `node2` the CU-CP and the CU-UP, `node10` the core and its database, `node11` the monitoring and the live CSI view. `sopnode-f3` and `sopnode-f4`, provisioned by POS, run the DUs (the `legacy_pos` provisioner remains declared but no node uses it). `FHI72_NODE_PARAMS` (after the deployment model) describes the O-RAN 7.2 setup of each RAN node: fronthaul PF and its VFs (one per C/U plane, with the DU MAC and the VLAN of its RU: 801 for `benetel1`, 802 for `benetel2`), housekeeping CPUs, NIC IRQ pinning and PTP. Both RAN nodes boot the PREEMPT_RT image with the same boot parameters.

For the radio hardware, we are going to use `benetel1` and `benetel2` as RUs in R2Lab, as well as `qhat01`, `qhat02` and `qhat03` as UEs in R2Lab.

The deployment model needs your [R2Lab](https://r2lab.inria.fr/tuto-010-registration.md) credentials (your R2Lab slice name, email, and password).

A good practice to prevent private information leak (e.g., password) is to store the sensitive information in a special location. In this, we keep it as environment variables that we store in a `.env` [file](https://pypi.org/project/python-dotenv/), next to this notebook or in a parent directory (e.g. the root of this repository, shared by all the notebooks; `load_dotenv()` takes the first one found going up from the notebook directory):

```
R2LAB_SLICE=<YOUR_R2LAB_SLICE>
R2LAB_EMAIL=<YOUR_R2LAB_EMAIL>
R2LAB_PASSWORD=<YOUR_R2LAB_PASSWORD>
```

The following lines load variables from this `.env` file. Make sure to never commit your secrets in public repositories.

In [ ]:
%pip install python-dotenv

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()
R2LAB_SLICE = os.getenv("R2LAB_SLICE")
R2LAB_EMAIL = os.getenv("R2LAB_EMAIL")
R2LAB_PASSWORD = os.getenv("R2LAB_PASSWORD")
missing = [name for name in ("R2LAB_SLICE", "R2LAB_EMAIL", "R2LAB_PASSWORD") if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Set {missing} in a .env file next to this notebook or in a parent directory")

We can now create the deployment model.

In [ ]:
from pathlib import Path

from ifran.models.deployment import DeploymentModel

# OAI images: weekly OAI tag of the gNB/DU and CU-UP images. From 2026.w45, a DU no longer aborts when a UE
# re-establishes with SRB1 only (OAI 36061c71ab: assertion in encode_cellgroup_config() after a failed F1
# handover). The csi metric of monitoring.yaml keeps this tag for the CSI image (r2labuser/oai-gnb-fhi72-csi).
# r2labuser images (R2Lab OAI CI): the target DU of an F1 handover schedules the UE with the CellGroup sent in the
# handover command (reconfigCellGroup) once its CFRA completes. With the upstream images, it keeps the CellGroup built
# without the UE capabilities (SRS on 1 port, UL rank 1), and with 4 PUSCH antenna ports the UE cannot send its
# RRCReconfigurationComplete on the target cell.
OAI_TAG = "2026.w46"
OAI_REGISTRY = "docker.io/r2labuser"

# VMs (BI provisioner): Kubernetes control plane and iperf server (node1), CU-CP and CU-UP (node2), core and
# database (node10), monitoring and live CSI view (node11). sopnode-f3 and sopnode-f4 (POS) run the DUs.
deployment_model = DeploymentModel.from_yaml(
f"""
params:
  provisioning:
    extra_path: ''
  k8s:
    master_node: node1
    local_kubeconfig_path: {Path(f".k8s-{EXPERIMENT}.conf").resolve()}
provisioners:
  bi:
    type: bi
    params:
      project: {PROJECT}
      experiment: {EXPERIMENT}
      infra: {INFRA}
      flavor: {VM_FLAVOR}
      create_experiment: true
      node_duration: {VM_DURATION}
  pos:
    type: pos
    params:
      attempt_reservation: false
  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <YOUR_SLICES_USERNAME>
  r2lab:
    type: r2lab
    params:
      r2lab_slice: {json.dumps(R2LAB_SLICE)}
      reservation_params:
        email: {json.dumps(R2LAB_EMAIL)}
        password: {json.dumps(R2LAB_PASSWORD)}
nodes:
  node1:
    provisioner: bi
    roles: []
  node2:
    provisioner: bi
    roles:
    - ran
  node10:
    provisioner: bi
    roles:
    - core
    - database
  node11:
    provisioner: bi
    roles: []
  sopnode-f3:
    provisioner: pos
    # du: the fhi72 playbook runs on the du_nodes group
    roles:
    - ran
    - du
    provisioning_params:
      # PREEMPT_RT kernel image, as with 5g_ansible (pos_rt_image)
      image: ubuntu-questing-rt@2025-10-29T19:34:30+00:00
      is_staged_image: true
      # 5g_ansible boot_parameters_rt_f3: housekeeping CPUs 0-5 (= fhi72 reserved_cpus), 40 x 1G hugepages
      boot_params: net.ifnames=1 biosdevname=0 isolcpus=managed_irq,6-63 nohz_full=6-63
        nohz=on rcu_nocbs=6-63 kthread_cpus=0-5 irqaffinity=0-5 rcu_nocb_poll
        selinux=0 enforcing=0 intel_pstate=disable nosoftlockup
        hugepagesz=1G hugepages=40 hugepagesz=2M hugepages=0 default_hugepagesz=1G
        mitigations=off intel_iommu=on processor.max_cstate=1 idle=poll
        intel_idle.max_cstate=0 iommu=pt skew_tick=1 tsc=nowatchdog
        nmi_watchdog=0 softlockup_panic=0 audit=0 mce=off nosmt
    # static CPU manager for the Guaranteed DU pod, system and Kubernetes processes on the housekeeping CPUs
    k8s_params:
      kubelet_params:
        cpu_manager_policy: static
        cpu_manager_policy_options:
          full-pcpus-only: 'true'
        reserved_system_cpus: 0-5
        topology_manager_policy: best-effort
  sopnode-f4:
    provisioner: pos
    # du: the fhi72 playbook runs on the du_nodes group
    roles:
    - ran
    - du
    provisioning_params:
      # PREEMPT_RT kernel image, as with 5g_ansible (pos_rt_image)
      image: ubuntu-questing-rt@2025-10-29T19:34:30+00:00
      is_staged_image: true
      # 5g_ansible boot_parameters_rt_f3 (same hardware as sopnode-f3): housekeeping CPUs 0-5 (= fhi72 reserved_cpus), 40 x 1G hugepages
      boot_params: net.ifnames=1 biosdevname=0 isolcpus=managed_irq,6-63 nohz_full=6-63
        nohz=on rcu_nocbs=6-63 kthread_cpus=0-5 irqaffinity=0-5 rcu_nocb_poll
        selinux=0 enforcing=0 intel_pstate=disable nosoftlockup
        hugepagesz=1G hugepages=40 hugepagesz=2M hugepages=0 default_hugepagesz=1G
        mitigations=off intel_iommu=on processor.max_cstate=1 idle=poll
        intel_idle.max_cstate=0 iommu=pt skew_tick=1 tsc=nowatchdog
        nmi_watchdog=0 softlockup_panic=0 audit=0 mce=off nosmt
    # static CPU manager for the Guaranteed DU pod, system and Kubernetes processes on the housekeeping CPUs
    k8s_params:
      kubelet_params:
        cpu_manager_policy: static
        cpu_manager_policy_options:
          full-pcpus-only: 'true'
        reserved_system_cpus: 0-5
        topology_manager_policy: best-effort
ran:
  implementation: oai
  components:
    # CU-CP and CU-UP on a VM: the CPUs of the RAN nodes are isolated for the DUs
    cucp:
      type: cu_cp
      node_name: node2
      image:
        repository: {OAI_REGISTRY}/oai-gnb
        image_tag: {OAI_TAG}
      advanced_config:
        useAdditionalOptions: "--telnetsrv --telnetsrv.shrmod ci --telnetsrv.listenport 9091"  # telnet defaults to 9090, the port of the metrics-parser sidecar in the same pod
    cuup:
      type: cu_up
      node_name: node2
      image:
        repository: {OAI_REGISTRY}/oai-nr-cuup
        image_tag: {OAI_TAG}
    du01:
      type: du
      image:
        repository: {OAI_REGISTRY}/oai-gnb-fhi72
        image_tag: {OAI_TAG}
      advanced_config:
        local_rf: 'no'
        useAdditionalOptions: --log_config.global_log_options level,nocolor,time
        tr_preference: raw_if4p5
        sl_ahead: 5
        max_pdschReferenceSignalPower: -27
        ssb_PositionsInBurst_Bitmap: 1
        initialDLBWPlocationAndBandwidth: 1099
        initialDLBWPcontrolResourceSetZero: 11
        initialDLBWPsearchSpaceZero: 0
        initialULBWPlocationAndBandwidth: 1099
        prach_msg1_FDM: 0
        zeroCorrelationZoneConfig: 0
        preambleReceivedTargetPower: -100
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB_PR: 4
        # n60 (instead of n64): keeps preambles 60-63 out of CBRA, for the handover CFRA preamble
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 14
        p0_NominalWithGrant: -96
        p0_nominal: -96
        pMax: 23
        preambleTransMax: 8
        powerRampingStep: 3
        msg3_DeltaPreamble: 2
        hoppingId: 0
        # 4T4R: 4 DL layers, 4 UL antenna ports
        pdsch_AntennaPorts_XP: 2
        pdsch_AntennaPorts_N1: 2
        pusch_AntennaPorts: 4
        do_CSIRS: 1
        do_SRS: periodic
        # MACRLCs
        pusch_TargetSNRx10: 150
        pucch_TargetSNRx10: 220
        pusch_RSSI_Threshold: -150
        pucch_RSSI_Threshold: -150
        dl_bler_target_lower: 0.15
        dl_bler_target_upper: 0.35
        ul_bler_target_lower: 0.15
        ul_bler_target_upper: 0.35
        pusch_FailureThres: 500
        ul_max_mcs: 28
        stats_max_ue: 32
        # L1s: tx_amp_backoff_dB and phase_compensation must match the benetel1 OAI profile (ru_config.cfg)
        prach_dtx_threshold: 140
        pucch0_dtx_threshold: 10
        pusch_dtx_threshold: 10
        max_ldpc_iterations: 15
        tx_amp_backoff_dB: 12
        phase_compensation: 0
        # thread-pool, L1/RU thread cores, fhi_72 dpdk_devices, system_core, io_core and worker_cores are set by
        # the DU pod from its allocated VFs and pinned CPUs
        fhi_72:
          dpdk_mem_size: 0
          du_addr:
          - 00:11:22:33:44:67
          - 00:11:22:33:44:66
          ru_addr:
          - 70:b3:d5:e1:5b:81
          - 70:b3:d5:e1:5b:81
          mtu: 9600
          file_prefix: fhi_72
          fh_config:
          - T1a_cp_dl: [419, 470]
            T1a_cp_ul: [285, 336]
            T1a_up: [294, 345]
            Ta4: [0, 200]
            ru_config:
              iq_width: 9
              iq_width_prach: 9
      node_name: sopnode-f3
      # Guaranteed pod (requests = limits, integer cpu): the static CPU manager pins its CPUs
      resources:
        define: true
        requests:
          nf:
            cpu: 14
            memory: 8Gi  # 5Gi: du02 OOMKilled during a handover (2026-11-19)
        limits:
          nf:
            cpu: 14
            memory: 8Gi  # 5Gi: du02 OOMKilled during a handover (2026-11-19)
    du02:
      type: du
      image:
        repository: {OAI_REGISTRY}/oai-gnb-fhi72
        image_tag: {OAI_TAG}
      advanced_config:
        local_rf: 'no'
        useAdditionalOptions: --log_config.global_log_options level,nocolor,time
        tr_preference: raw_if4p5
        sl_ahead: 5
        max_pdschReferenceSignalPower: -27
        ssb_PositionsInBurst_Bitmap: 1
        initialDLBWPlocationAndBandwidth: 1099
        initialDLBWPcontrolResourceSetZero: 11
        initialDLBWPsearchSpaceZero: 0
        initialULBWPlocationAndBandwidth: 1099
        prach_msg1_FDM: 0
        zeroCorrelationZoneConfig: 0
        preambleReceivedTargetPower: -100
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB_PR: 4
        # n60 (instead of n64): keeps preambles 60-63 out of CBRA, for the handover CFRA preamble
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 14
        p0_NominalWithGrant: -96
        p0_nominal: -96
        pMax: 23
        preambleTransMax: 8
        powerRampingStep: 3
        msg3_DeltaPreamble: 2
        hoppingId: 0
        # 4T4R: 4 DL layers, 4 UL antenna ports
        pdsch_AntennaPorts_XP: 2
        pdsch_AntennaPorts_N1: 2
        pusch_AntennaPorts: 4
        do_CSIRS: 1
        do_SRS: periodic
        # MACRLCs
        pusch_TargetSNRx10: 150
        pucch_TargetSNRx10: 220
        pusch_RSSI_Threshold: -150
        pucch_RSSI_Threshold: -150
        dl_bler_target_lower: 0.15
        dl_bler_target_upper: 0.35
        ul_bler_target_lower: 0.15
        ul_bler_target_upper: 0.35
        pusch_FailureThres: 500
        ul_max_mcs: 28
        stats_max_ue: 32
        # L1s: tx_amp_backoff_dB and phase_compensation must match the benetel1 OAI profile (ru_config.cfg)
        prach_dtx_threshold: 140
        pucch0_dtx_threshold: 10
        pusch_dtx_threshold: 10
        max_ldpc_iterations: 15
        tx_amp_backoff_dB: 12
        phase_compensation: 0
        # thread-pool, L1/RU thread cores, fhi_72 dpdk_devices, system_core, io_core and worker_cores are set by
        # the DU pod from its allocated VFs and pinned CPUs
        fhi_72:
          dpdk_mem_size: 0
          du_addr:
          - 00:11:22:33:44:67
          - 00:11:22:33:44:66
          ru_addr:
          - 70:b3:d5:e1:5b:81
          - 70:b3:d5:e1:5b:81
          mtu: 9600
          file_prefix: fhi_72
          fh_config:
          - T1a_cp_dl: [419, 470]
            T1a_cp_ul: [285, 336]
            T1a_up: [294, 345]
            Ta4: [0, 200]
            ru_config:
              iq_width: 9
              iq_width_prach: 9
      node_name: sopnode-f4
      # Guaranteed pod (requests = limits, integer cpu): the static CPU manager pins its CPUs
      resources:
        define: true
        requests:
          nf:
            cpu: 14
            memory: 8Gi  # 5Gi: du02 OOMKilled during a handover (2026-11-19)
        limits:
          nf:
            cpu: 14
            memory: 8Gi  # 5Gi: du02 OOMKilled during a handover (2026-11-19)
rus:
  benetel1:
    vendor: benetel
    model: ran550
    provisioner: r2lab
    # no Python on the RU: the benetel playbook only runs raw commands there
    ssh_info:
      extra_ansible_vars:
        ansible_python_interpreter: ''
    # management IP, also returned by the radio API once the RU is ready
    ips:
    - 192.168.233.101
  benetel2:
    vendor: benetel
    model: ran550
    provisioner: r2lab
    # no Python on the RU: the benetel playbook only runs raw commands there
    ssh_info:
      extra_ansible_vars:
        ansible_python_interpreter: ''
    ips:
    - 192.168.233.102
ues:
  qhat01:
    type: cots
    provisioner: r2lab
  qhat02:
    type: cots
    provisioner: r2lab
  qhat03:
    type: cots
    provisioner: r2lab
"""
)

# O-RAN 7.2 nodes and Benetel RUs, outside IFRAN: miscellaneous field of the deployment model, read by the fhi72 and
# benetel playbooks of this repository (ansible/) and by its OAI charts (radio fhi72, SR-IOV resources, 1G hugepages)
from r2lab_experiments.fhi72 import BenetelRUParams, Fhi72NodeParams, fhi72_miscellaneous


def fhi72_node(fronthaul_interface: str, vlan: int, ptp_interface: str) -> Fhi72NodeParams:
    # DU MAC addresses and VLAN of the VFs, also loaded in the RU (/etc/ru_config.cfg)
    return Fhi72NodeParams.model_validate({
        "fronthaul_interface": fronthaul_interface,
        "vfs": [
            {"resource_name": "ruvfiou", "plane": "U", "mac": "00:11:22:33:44:66", "vlan": vlan},
            {"resource_name": "ruvfioc", "plane": "C", "mac": "00:11:22:33:44:67", "vlan": vlan},
        ],
        "reserved_cpus": "0-5",
        "irq_pattern": "ice-|ens15f[0-3]np[0-3]",
        "ptp": {"interface": ptp_interface},
    })


FHI72_NODE_PARAMS = {
    "sopnode-f3": fhi72_node("ens15f1np1", 801, "ens15f2np2"),
    "sopnode-f4": fhi72_node("ens15f2np2", 802, "ens15f3np3"),  # sw5 Ethernet1/37
}
BENETEL = BenetelRUParams(tx_power_dbm=24, downlink_scaling=0)
# RU -> RAN node of its DU (which reaches its management IP), DU, parameters
BENETEL_RUS = {"benetel1": ("sopnode-f3", "du01", BENETEL), "benetel2": ("sopnode-f4", "du02", BENETEL)}
deployment_model.miscellaneous = fhi72_miscellaneous(metamodel, FHI72_NODE_PARAMS, BENETEL_RUS, hugepages_1gi="8Gi")

> *IMPORTANT NOTE*: Make sure your R2Lab credentials are not shared publicly!

By default, the metamodel and deployment model will use Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
and one 5G slice (`internet`) with one DNN (`internet`).

## **1.**   Provisioning nodes:

In [ ]:
from ifran.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision_nodes()

## **2.**   Kubernetes deployment:

In [ ]:
from ifran.deployment import K8SManagement

k8s = K8SManagement(prov)

In [ ]:
await k8s.deploy()

### R2Lab netplan of sopnode-f3 and sopnode-f4

As with 5g_ansible: DHCP on the PTP interface (`ptp`, which also carries the RU management network 192.168.233.0/24) gives `sopnode-f3` and `sopnode-f4` access to the RUs (`ssh benetel1`, `ssh benetel2`), and the USRP and AW2S VLANs on the fronthaul PF (`rrus`). Only the netplan role of the r2lab example tuning is applied: the fhi72 setup below replaces its NIC and CPU tuning. Apply it before the fhi72 setup, as `netplan apply` resets the interfaces.

In [ ]:
from pathlib import Path

from ifran.deployment import MiscellaneousManagement

misc = MiscellaneousManagement(k8s)
await misc.run_playbook(
    playbook=str(Path("netplan.yaml").resolve()),
    roles_paths=[str(Path("../oai-aw2s/tuning/ansible/roles").resolve())],
)


### O-RAN 7.2 fronthaul setup of sopnode-f3 and sopnode-f4

Once the RAN nodes are in the cluster: VFs created on the fronthaul PF (`ens15f1np1` on `sopnode-f3`, `ens15f2np2` on `sopnode-f4`) and bound to `vfio-pci`, NIC IRQs pinned on the housekeeping CPUs, CPU governor, RCU threads, sysctl, SR-IOV device plugin (resources `openshift.io/ruvfioc` and `openshift.io/ruvfiou`) and PTP (`ptp4l` slave on `ens15f2np2` / `ens15f3np3`, domain 24, and `phc2sys`), by the `fhi72` playbook of this repository (`ansible/`), with the `fhi72_params` of `miscellaneous`. The kubelet static CPU manager (`full-pcpus-only`) is set when the nodes join the cluster (`k8s_params` of the nodes).

In [ ]:
# O-RAN 7.2 setup of the du_nodes, with the fhi72_params of the miscellaneous field
await misc.run_playbook(
    playbook=str(REPOSITORY / "ansible" / "playbooks" / "fhi72.yaml"),
    roles_paths=[str(REPOSITORY / "ansible" / "roles")],
)


Check each RAN node: 40 free 1G hugepages, both VFs with their MAC and VLAN (801 on `sopnode-f3`, 802 on `sopnode-f4`) bound to `vfio-pci`, `ptp4l` and `phc2sys` active, and a PTP grandmaster present with a small `master_offset` (ns).

In [ ]:
# Fronthaul node state: 1G hugepages, VFs (MAC/VLAN) bound to vfio-pci, PTP (ptp4l slave, phc2sys)
FHI72_NODES = {"sopnode-f3": "ens15f1np1", "sopnode-f4": "ens15f2np2"}  # RAN node: fronthaul PF
for node, pf in FHI72_NODES.items():
    check = await prov.run_node_command(
        command=["bash", "-c", " ; ".join([
            "grep -E 'HugePages_(Total|Free)|Hugepagesize' /proc/meminfo",
            f"ip -d link show {pf} | grep -E '^ +vf '",
            f"for vf in /sys/class/net/{pf}/device/virtfn*; do "
            "echo $(basename $(readlink $vf)) $(basename $(readlink $vf/driver)); done",
            "systemctl is-active ptp4l phc2sys",
            "/usr/local/sbin/pmc -u -b 0 -d 24 'GET TIME_STATUS_NP' | grep -E 'master_offset|gmPresent|gmIdentity'",
            "true",  # print every check, even when one fails
        ])],
        nodes=[node],
        become=True,
        quiet=True,
    )
    print(f"--- {node} ({pf})\n{check}")


In [ ]:
# The SR-IOV device plugin must advertise one VF per resource on each RAN node, and the kubelet the 1G hugepages
for node in FHI72_NODES:
    allocatable = await prov.run_node_command(
        command=["kubectl", "--kubeconfig", "/etc/kubernetes/admin.conf", "get", "node", node,
                 "-o", "jsonpath={.status.allocatable}"],
        nodes=["node1"],
        become=True,
        quiet=True,
    )
    print(f"{node}: {allocatable}")


## **3.**   5G core deployment:

In [ ]:
from ifran.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## **4.**   RAN deployment:

First, we are going to provision the remaining resources (RUs and UEs):

IFRAN powers on the UEs and the RUs (`benetel1`, `benetel2`) through the radio API, then waits until the API reports each RU ready. The API checks that an RU answers ping on its management network (192.168.233.0/24 for the Benetel RUs): the radio initialization of the RU is checked later by the `benetel` playbook.

In [ ]:
prov_radio = ProvisioningManagement(core)
await prov_radio.provision()

### Then we are going to deploy the OAI RAN control-plane and the first DU (`du01` with `benetel1`).

Each DU pod requests the two VFs and the 1G hugepages of its node, and pins its threads on the CPUs allocated by the static CPU manager.

Before the DUs, the `benetel` playbook of this repository loads the OAI configuration of `benetel1` and `benetel2` (`/etc/ru_config.cfg`, the `ru_config` of `miscellaneous`) over SSH at their management IP, through the RAN node of their DU (raw commands: the RUs have no Python): DU MAC addresses and VLAN of the VFs of that node, centre frequency, bandwidth and TDD pattern of the DU radio configuration, MIMO mode of the RU ports, and the `BENETEL` parameters. The RUs are then rebooted (a few minutes), and the playbook waits for their radio initialization.

In [ ]:
from ifran.deployment import MiscellaneousManagement, RANManagement

# Benetel RUs: the benetel playbook reaches each RU over SSH at its management IP (provisioned address), through
# the RAN node of its DU (jumphost). The IP is then removed from the RAN model: the OAI radio derivation of IFRAN
# takes an RU IP for a USRP address (sdr_addrs), while these RUs use the O-RAN 7.2 fronthaul.
misc_ru = MiscellaneousManagement(preceding=prov_radio)
for ru_name, (node_name, _, _) in BENETEL_RUS.items():
    ru = misc_ru.deployment_model.rus[ru_name]
    ru.ssh_info.hostname = str(ru.ips[0])
    ru.ssh_info.jumphost = misc_ru.deployment_model.nodes[node_name].ssh_info
for ru_name in BENETEL_RUS:
    misc_ru.deployment_model.rus[ru_name].ips = []

ran = RANManagement(misc_ru)

In [ ]:
import asyncio

from ifran.deployment import UEManagement

# The UEs are configured (provisioner) while the RUs load their configuration: both are independent and take
# minutes. The UE manager of the UE section starts from the RAN model, so the configured state is copied there.
ue_setup = UEManagement(preceding=ran)
await asyncio.gather(
    misc_ru.run_playbook(
        playbook=str(REPOSITORY / "ansible" / "playbooks" / "benetel.yaml"),
        roles_paths=[str(REPOSITORY / "ansible" / "roles")],
        retries=1,  # an RU reboot takes minutes: no retry
    ),
    ue_setup.configure(ues=["qhat01", "qhat02"]),
)
for name, ue_prov in ue_setup.deployment_model.ues.items():
    ran.deployment_model.ues[name].connection_status = ue_prov.connection_status


In [ ]:
# OAI charts of this repository: IFRAN's charts (fhi72 branch, their O-RAN 7.2 values from miscellaneous) with the
# NADOR metrics-parser sidecar (2026.w46 UE header) and CSI file. NADOR keeps a chart set before configure_ran_monitoring.
from r2lab_experiments import OAI_CHARTS

oai_charts = OAI_CHARTS
for name, component in ran.deployment_model.ran.components.items():
    if (
        component.implementation == "oai"
        and component.type in ("gnb", "cu", "cu_cp", "cu_up", "du")
        and (name in ['cucp', 'cuup'] or component.type == "du")
    ):
        component.custom_chart_path = str(oai_charts / f"oai-{component.type}")
        component.custom_rendered_files = [
            str(oai_charts / "values.yaml.j2"),
            str(oai_charts / "config.yaml.j2"),
        ]


### Monitoring profile and CSI logging

`monitoring.yaml` sets what the monitoring collects. Its `ran_kpis` and `csi` metrics change the RAN, so the profile is read now and `configure_ran_monitoring` applies them before the RAN is deployed:

- `csi`: `du01` (`benetel1` on `sopnode-f3`) logs SRS channel estimates per resource block. `du02` could too, but `sopnode-f4` has no data disk for the CSI file (its disk holds a Proxmox installation), and the CU-CP and CU-UP have no PHY, hence no SRS. The `csi` metric of `monitoring.yaml` (`components: [du01]`) turns the logger on: `du01` gets a CSI logger image with the same OAI tag (every other OAI component uses the image without logger), periodic SRS and CSI-RS in the rendered configuration, and `/data/csi` mounted from the node: each logger writes `/data/csi/csi_per_rb.csv` on its own node, where the file survives pod restarts. The logger options (`granularity`, `subcarrier_sampling`, `antennas`, `ports`) are those of the `csi` metric: see the [CSI guide](../../docs/guide/csi.md).

With a real radio the file grows by several MB/s: experiments copy only the per-section files (the `CsiCollector` options below), still reading the whole file on the radio node at the lowest priority, on its housekeeping CPUs. Run these cells before preparing the radios.

#### What the CSI is

The CSI logged here is the raw uplink channel `H` that the gNB estimates on the sounding reference signal (SRS) of each UE: one complex value (amplitude and phase) per resource block, receive antenna and SRS port, at every SRS occasion. A CSI logger patched into the PHY of the gNB copies each estimate into memory, and a separate thread writes it every 5 seconds to a CSV file on the radio node. NADOR shows this file live and collects it per experiment section.

The [CSI guide](../../docs/guide/csi.md) explains why the SRS is used, how the OAI and OCUDU gNBs extract it, what NADOR deploys, shows and collects, the tools that read the files, and the limits of the approach.

In [ ]:
from nador import MonitoringDeployment, configure_ran_monitoring

# The same profile is used below to deploy the monitoring
config = MonitoringDeployment.from_yaml_file("monitoring.yaml")
# NADOR 1.1.1 knows no CSI image for the O-RAN 7.2 OAI image: name it for the radios that log
from r2lab_experiments import name_oai_fhi72_csi_images

print("CSI images named:", name_oai_fhi72_csi_images(ran, config))
configure_ran_monitoring(ran, config)

# RAN components that log CSI (the csi metric of monitoring.yaml)
CSI_COMPONENTS = [
    name for name, component in ran.deployment_model.ran.components.items()
    if str((component.extra_env_vars or {}).get("CSI_LOGGING_ENABLED")) == "1"
]
for name in CSI_COMPONENTS:
    component = ran.deployment_model.ran.components[name]
    print(f"{name} on {component.node_name}: {component.image.repository}:{component.image.image_tag}",
          component.extra_env_vars)

#### Default CSI settings

With the `csi` metric of `monitoring.yaml` as provided, the settings below apply. Change them where the last column says; logger settings take effect when the radios start, so set them before preparing the radios.

| Setting | Value in this notebook | Where it is set |
|---|---|---|
| Granularity (`granularity`) | `rb`: one complex value per RB | `csi` metric (`monitoring.yaml`) |
| Subcarrier sampling (`subcarrier_sampling`) | 1 (only used with `subcarrier`) | `csi` metric (`monitoring.yaml`) |
| Antennas / ports (`antennas` / `ports`) | all | `csi` metric (`monitoring.yaml`) |
| Writer thread CPU (`CSI_FLUSH_CORE`) | last CPU of the DU pod, removed from the L1 thread pool (O-RAN 7.2 DU pod) | nador OAI DU chart |
| SRS period | chosen by OAI: smallest allowed period covering TDD period x maximum number of UEs per gNB (measured in the rfsim scenarios: 160 slots, 80 ms) | OAI gNB/DU (`do_SRS: periodic`) |
| CSI file | `/data/csi/csi_per_rb.csv` on the radio node (hostPath) | `csi` metric (`output_dir`) |
| CSI storage | data disk bind-mounted on `/data/csi` (`CSI_STORAGE` partitions) | `prepare_csi_storage` |
| Whole CSI file copied per run (`fetch_full`) | no (per-section files only) | `CSI_COLLECTORS` (`CsiCollector`) |
| Windows cut (`window_types`, `padding_seconds`) | `section`, 0 s | `CSI_COLLECTORS` (`CsiCollector`) |
| Reading CPUs on the radio node (`cpu_list`) | `0-4` (housekeeping CPUs) | `CSI_COLLECTORS` (`CsiCollector`) |
| Temporary files on the radio node (`cleanup_remote`) | removed | `CsiCollector` default |
| Clock offset correction | applied only above 0.5 s (offset and drift recorded per run) | fixed (`nador.csi.collection.CLOCK_THRESHOLD_S`) |
| Live view | on `node11`, port `8501`, 120 s window, 10 s refresh, live copy restarted beyond 1024 MB, visualizer v9.0 (`oai-csi-logging` `csi-v3.1`), `tail -F` on CPUs `0-4` | `CSI_LIVE`, `CSI_LIVE_PORTS` |

The R2Lab nodes boot a live image: their root filesystem is in RAM, where the CSI file would fill the memory
(5g_ansible saw the gNB pod killed after about 2 minutes). `prepare_csi_storage` bind-mounts the `csi-data`
directory of the node's data disk onto `/data/csi`, as 5g_ansible does; the partitions are those of the 5g_ansible
inventory (`storage=`). The disk is never formatted, and the mount does not survive a reboot: rerun this cell after
reprovisioning. It stops with an explicit error if a node's directory is in RAM and no partition is given.

In [ ]:
from nador.csi import prepare_csi_storage

# Data disk partition (under /dev) of each radio node whose root is in RAM; nodes on a disk need no entry
CSI_STORAGE = {'sopnode-f3': 'sdb2'}  # 5g_ansible deploy.sh get_storage
for directory, report in (await prepare_csi_storage(ran, CSI_STORAGE)).items():
    print(f"{directory}: {report['before']}  ->  {report['after']}")

In [ ]:
await ran.prepare(components=['cucp', 'cuup'])

In [ ]:
await ran.install(components=['cucp', 'cuup'])

In [ ]:
await ran.deploy(components=['du01'])

Check logs of `du01`:

In [ ]:
print(await ran.get_logs(quiet=True))

The DU is deployed and ready for connections.

## **5.**   Connecting UEs:

## Monitoring before UE attachment

Monitoring runs on the VM **node11**: the CPUs of the RAN nodes are isolated for the DUs. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

# config: the monitoring profile read before the RAN deployment (monitoring.yaml)
monitoring = MonitoringManagement(preceding=ran, config=config)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display
from ifran.models.provisioning import SSHInfo
from ifran.utils.utils import deep_merge

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
# global SSH defaults merged with the SSH info of the node, as for Ansible
ssh = SSHInfo.model_validate(deep_merge(
    monitoring.deployment_model.params.ssh.model_dump(mode="json", exclude_unset=True),
    monitoring.deployment_model.nodes[control].ssh_info.model_dump(mode="json", exclude_unset=True),
))

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.get_common_args() or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

### Live CSI view

One Streamlit view per CSI radio runs on `node11`, never on a radio node: it follows the radio node's CSI
file over SSH, where only `sshd` and `tail -F` run, at the lowest priority on the housekeeping CPUs `0-4`. The view
shows a sliding window (120 s by default) in local time. Start it once the radio runs (it reads the file header);
the tunnel is opened on this machine. `du02` has no CSI logger here (see above).

In [ ]:
from nador.csi import CsiLiveView
from IPython.display import Markdown, display

# Live view on node11, never on a radio node; housekeeping CPUs for "tail -F" on the radio node
CSI_LIVE = {"window_s": 120, "refresh_s": 10, "max_mb": 1024, "cpu_list": "0-4"}
CSI_LIVE_PORTS = {'du01': 8501, 'du02': 8502}
lives = globals().get("lives", {})
for name in CSI_COMPONENTS:
    if name in lives:
        await lives[name].close_tunnel()  # re-run: free the local port of the previous tunnel
    lives[name] = CsiLiveView(ran, live_node='node11', component=name, port=CSI_LIVE_PORTS[name], **CSI_LIVE)
    url = await lives[name].start()
    display(Markdown(f"[CSI {name}]({url})"))

In [ ]:
from ifran.deployment import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
# Handover scenario: qhat01, then qhat02, on du01, one at a time. The CU UE ID of qhat01 is the UE context with a
# PDU session (DRB) created after its connection (next cells): another UE of the room may hold CU UE ID 1.
# qhat03 stays off: it does not attach to du01 alone, and benetel2 saturates it.
from nador.handover.oai import connected_ues

HO_UES = ["qhat01", "qhat02"]
# qhat03 keeps its radio on from a previous experiment and attaches by itself as soon as du01 is up (CU UE ID 1,
# without PDU session, 2026-11-19): turn its radio off
await ue.disconnect(ues=["qhat03"])
before = {u["ue"]: u["rnti"] for u in connected_ues(await ran.get_logs(components=["cucp"], quiet=True))}
await ue.connect(ues=["qhat01"])

Find the CU UE ID of `qhat01`: the UE context of the CU-CP created since its connection, on `du01` (cell 1), with a PDU session (DRB: OAI hands over only a UE with a DRB). Then connect `qhat02`:

In [ ]:
ues = connected_ues(await ran.get_logs(components=["cucp"], quiet=True))
for u in ues:
    print(f"CU UE ID {u['ue']}: RNTI {u['rnti']}, cell {u['cell']}, {'PDU session' if u['drb'] else 'no PDU session'}")
new = [u for u in ues if before.get(u["ue"]) != u["rnti"] and u["drb"] and u["cell"] == 1]
if len(new) != 1:
    raise RuntimeError(f"expected one new UE with a PDU session on du01 for qhat01, found {new}: connect qhat01 again")
HO_UE_ID = new[0]["ue"]  # CU UE ID of qhat01, handed over in section 7
print(f"qhat01: CU UE ID {HO_UE_ID}")

In [ ]:
await ue.connect(ues=["qhat02"])

Check `du01` logs after UEs were connected:

In [ ]:
print(await ran.get_logs(quiet=True))

`qhat01` and `qhat02` are connected to the 5G network.

Let's verify their connectivity by pinging the UPF:

In [ ]:
ping = await prov_radio.run_ue_command(
    command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], ues=HO_UES, quiet=True, flatten=False
)

In [ ]:
print(ping["qhat01"])

In [ ]:
print(ping["qhat02"])

## **6.**   Deploy the second DU (`du02` with `benetel2`):

In [ ]:
ran = RANManagement(ue)

In [ ]:
await ran.deploy(components=['du02'])

If `du02` is in `CSI_COMPONENTS`, start its live CSI view, then check that the DU loggers write their files:

In [ ]:
if "du02" in CSI_COMPONENTS:
    if "du02" in lives:
        await lives["du02"].close_tunnel()  # re-run: free the local port of the previous tunnel
    lives["du02"] = CsiLiveView(ran, live_node="node11", component="du02", port=CSI_LIVE_PORTS["du02"], **CSI_LIVE)
    display(Markdown(f"[CSI du02]({await lives['du02'].start()})"))

Check that each CSI logger runs and that its file grows on disk: the logs show the logger start, and each
`csi_per_rb.csv` must grow between the two listings (the logger flushes every 5 seconds); `df` shows the space left.

In [ ]:
from ifran.deployment import ProvisioningManagement

for name in CSI_COMPONENTS:
    logs = await ran.get_logs(components=[name], quiet=True)
    print(f"--- {name}")
    print("\n".join(line for line in logs.splitlines() if "[CSI]" in line or "Init: N_RB_DL" in line))

csi_nodes = {name: ran.deployment_model.ran.components[name].node_name for name in CSI_COMPONENTS}
nodes = ProvisioningManagement(preceding=ran)
for _ in range(2):  # sizes must grow between the two listings
    listing = await nodes.run_node_command(
        command=["sh", "-c", "ls -l /data/csi/csi_per_rb.csv; df -h /data/csi | tail -1"],
        nodes=sorted(set(csi_nodes.values())), become=True, quiet=True, flatten=False,
    )
    for name, node in csi_nodes.items():
        print(f"{name} ({node}):\n{listing[node].strip()}")
    await asyncio.sleep(10)

#### Check `du01`'s and `du02`'s logs before the F1-handover attempt:

In [ ]:
logs = await ran.get_logs(components=["du01", "du02"], quiet=True)

## **7.**   Trigger F1-Handover using OAI CU-CP's telnet server command:

#### What happens during an F1 handover

`du01` and `du02` are connected over F1 to the same CU-CP, which keeps the UE context (RRC, PDCP, PDU sessions): the core network sees nothing, only the lower layers of the UE (RLC, MAC, PHY) move from one DU to the other. `run_handover` checks the steps of the procedure (3GPP TS 38.401, section 8.2.1.1) in the CU-CP and DU logs:

1. **Decision**: here, a command sent to the CU-CP.
2. **Preparation of the target DU** (F1AP UE context setup): new C-RNTI and, with contention-free random access, a dedicated preamble.
3. **Handover command**: RRCReconfiguration with reconfigurationWithSync, sent to the UE through the source DU.
4. **Random access on the target cell**: the UE sends its preamble (Msg1).
5. **Random access response** (Msg2) of the target DU: a UE that does not decode it sends its preamble again.
6. **Completion**: the CU-CP receives RRCReconfigurationComplete from the target DU, the only proof that the UE reached the target cell (success).
7. **Release** of the UE on the source DU (F1AP UE context release).

Guides: [the F1 handover procedure](../../docs/guide/handover-procedure.rst), [how OAI implements and logs it](../../docs/guide/handover-oai.rst), [the handover report of NADOR](../../docs/guide/handover.md), [how the notebooks trigger it](../../docs/guide/handover-trigger.rst).

Get one of the UE's CU UE ID (usually they are incremental integers starting from 1)

In [ ]:
print(await ran.get_logs(components=["cucp"], quiet=True))

`run_handover` triggers the handover of a UE (CU UE ID, above) from `du01` to `du02` with the CU-CP telnet command `ci trigger_f1_ho`, waits 15 s, then checks each step in the CU-CP and DU logs: trigger, preparation of the UE on the target DU (dedicated CFRA preamble), reception of this preamble, random access response (and whether the UE sends the preamble again: it did not decode the response), and `RRCReconfigurationComplete` on the target cell. It reports what followed a failure (re-establishments, rollback, new attachment, DU abort or restart, whose previous logs are then read), where the UE is now served, and the ping of its COTS UE.

`HO_UE_ID` is the CU UE ID of `qhat01`, found after its connection (section 5), and `HO_UE` its COTS UE for the ping. The UE must be served by `du01` with a clean link, and receive `du02` without saturation (RSRP of about -70 to -90 on both cells).

In [ ]:
from r2lab_experiments.handover import run_handover

HO_UE = "qhat01"  # COTS UE of HO_UE_ID (ping after the handover)
handover = await run_handover(
    ran, HO_UE_ID, source="du01", target="du02",
    ue_manager=ProvisioningManagement(preceding=ue), ue_name=HO_UE, ue_interface="wwan0",
)

Reference: https://github.com/OPENAIRINTERFACE/openairinterface5g/blob/develop/doc/handover-tutorial.md

With the OAI image used here (`f80a883`), reading only the first line of the answer and closing crashed the CU-CP: the connection is reset because of the unread rest of the answer, `recv()` then returns -1 in the telnet server (`common/utils/telnetsrv/telnetsrv.c`, `if(!readc) break;` does not catch it), and the next `recv()` aborts with `*** buffer overflow detected ***` (exit code 134). The handover was therefore interrupted by the CU-CP restart (the target DU drops the prepared UE context, the UEs re-attach on either DU). The command above reads the whole answer before closing. Check the CU-CP with `kubectl -n oai-ran get pods` (restarts) and `kubectl -n oai-ran logs <cucp pod> -c cucp --previous`, then the DU serving the UE with `get_logs` below.

## Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline. The run also collects the CSI of each radio that logs it, cut per section on its node and saved under `csi/<component>/`.

In [ ]:
from datetime import UTC, datetime
from ifran.deployment import ProvisioningManagement
from nador.artifacts import ExperimentRun
from nador.csi.collection import CsiCollector
from nador.traffic import Iperf

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = HO_UES  # qhat03 is not connected in this scenario
UE_INTERFACE = "wwan0"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

# COTS UEs over SSH: iperf3 installed where missing, address of each UE on UE_INTERFACE
clients = ProvisioningManagement(preceding=ue)
iperf = await Iperf.setup(clients, server=SERVER_NODE, interface=UE_INTERFACE, clients=CLIENTS)
print(f"Server: {SERVER_NODE} ({iperf.server_ip}); UE clients: {iperf.ue_ips}")

In [ ]:
# CSI retrieval by each ExperimentRun: the CSI file is cut per section on the radio node, at the lowest priority on
# the housekeeping CPUs, and only the cuts are fetched (the whole file grows by several MB/s)
CSI_COLLECTORS = [CsiCollector(monitoring, name, fetch_full=False, cpu_list="0-4") for name in CSI_COMPONENTS]

In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'qhat01', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'qhat01', "direction": "downlink", "duration": DURATION},
    {"name": "ue2_uplink", "client": 'qhat02', "direction": "uplink", "duration": DURATION},
    {"name": "ue2_downlink", "client": 'qhat02', "direction": "downlink", "duration": DURATION},
    {"name": "ue3_uplink", "client": 'qhat03', "direction": "uplink", "duration": DURATION},
    {"name": "ue3_downlink", "client": 'qhat03', "direction": "downlink", "duration": DURATION},
]
SCENARIO = [step for step in SCENARIO if step["client"] in CLIENTS]  # connected UEs only

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    collectors=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    with experiment.section("ping"):
        ping = await clients.run_ue_command(
            command=["ping", "-c", "6", "11.1.1.1", "-I", UE_INTERFACE], ues=CLIENTS, quiet=True, flatten=False
        )
    experiment.save_json("ping.json", ping)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await iperf.serve(port)
        with experiment.section(test["name"]):
            report = await iperf.client(test["client"], test["direction"], test["duration"], port)
        # a failed test returns a report with an error key: save it and go on with the next test
        if report.get("error"):
            print(f"FAILED {test['name']}: {report['error']}")
        experiment.save_json(f"{test['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")


In [ ]:
from nador.artifacts import iperf_summary

print(iperf_summary(output_directory))

Check the CSI collected by the run: per radio, its node clock (the offset to this machine is only applied above
0.5 s), the rows and 5-second batches read in the whole file, and the per-section files under
`csi/<component>/by_window/section/`.

In [ ]:
meta = json.loads((output_directory / "experiment_metadata.json").read_text())
print(meta["status"], meta["collection_errors"])
for name, m in meta.get("csi", {}).items():  # empty without CSI
    print(f"--- {name} on {m['node']}")
    print("  clock:", {k: m.get(k) for k in ("offset_applied_s", "clock_drift_s", "warnings")})
    print("  start:", {k: round(m["clock_start"][k], 3) for k in ("offset_s", "rtt_s", "uncertainty_s")})
    print("  data: ", {k: m.get("summary", {}).get(k) for k in ("rows", "markers", "dropped_rows", "windows_written", "windows_empty")})
print(sorted(str(p.relative_to(output_directory)) for p in (output_directory / "csi").rglob("*.gz")))

### End of the experiment

**Run All** stops at the next cell, so that the end-of-experiment cells below (stop of the live CSI views, UE
disconnection, release of the nodes) only run when you run them yourself, one at a time. The kernel keeps its state:
run the cells below with Shift+Enter when the experiment is over.

In [ ]:
class StopRunAll(Exception):
    """Ends "Run All" here; IPython shows only the message below (no traceback)."""

    def _render_traceback_(self):
        return [str(self)]


raise StopRunAll("Run All stops here. When the experiment is over, run the cells below one at a time.")

Stop the live CSI views before releasing the nodes: this closes the tunnels, stops the viewers and revokes their
SSH keys on the radio nodes.

In [ ]:
for view in globals().get("lives", {}).values():
    await view.stop()

Let's disconnect the UEs for now:

In [ ]:
ue = UEManagement(ran)
await ue.disconnect(force=True)

And finally save this experiment's meta model and deployment model:

In [ ]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/ifran/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.

## Cleanup — only after the experiment

Run the next cell **after** the experiment (and after stopping the live CSI views) to power off the RUs and the UEs (R2Lab radio API), free the POS nodes (allocations freed, calendar events kept) and destroy the VMs. The cell does nothing until you set `CONFIRM_CLEANUP = True` in it: the VMs hold the Kubernetes control plane, the CUs, the core and its database, and the monitoring.

In [ ]:
# True: power off the RUs and the UEs, free the POS nodes, destroy the VMs (end of the experiment)
CONFIRM_CLEANUP = False
if not CONFIRM_CLEANUP:
    raise RuntimeError(
        "Cleanup not confirmed: set CONFIRM_CLEANUP = True in this cell to power off the RUs and UEs, free the POS"
        f" nodes and destroy {VM_NODES}"
    )
off = ProvisioningManagement(preceding=ue)
failed = {}
# RUs and UEs: powered off through the R2Lab radio API
for kind, power_off in (("RUs", off.unprovision_rus), ("UEs", off.unprovision_ues)):
    try:
        await power_off(quiet=True)
        print(f"{kind} powered off: {list(getattr(off.deployment_model, kind.lower()))}")
    except Exception as exc:  # noqa: BLE001 - go on with the rest, then fail
        failed[kind] = exc
# POS nodes, one at a time: allocation freed (slices pos allocations free <node> --keep-calendar-event)
for node, data in off.deployment_model.nodes.items():
    if data.provisioner != "pos" or not data.is_provisioned:
        continue
    try:
        await off.unprovision_nodes(nodes=[node], quiet=True)
        print(f"{node}: freed")
    except Exception as exc:  # noqa: BLE001 - report every node, then fail
        failed[node] = exc
# VMs, one at a time: destroyed in parallel, the "slices project use" calls of IFRAN race on the cache of the slices
# CLI (~/.slices/cache) and fail with FileNotFoundError. A destroyed VM is skipped: rerun the cell after a failure.
for node in VM_NODES:
    if not off.deployment_model.nodes[node].is_provisioned:
        print(f"{node}: already destroyed")
        continue
    try:
        await off.unprovision_nodes(nodes=[node], quiet=True)
        print(f"{node}: destroyed")
    except Exception as exc:  # noqa: BLE001 - report every VM, then fail
        failed[node] = exc
if failed:
    raise RuntimeError(f"Cleanup incomplete (run the cell again): {failed}")